# BE5210 Final Project — Test Set Prediction

**Model:** AutoEncoder1D (1D U-Net CNN), trained as CNN v4 (no pretraining, all skip connections).

## Instructions

1. Upload `algorithm.zip` and `truetest_data.mat` to this Colab session (Files panel on the left).
2. Run all cells top to bottom (**Runtime → Run all**).
3. The notebook outputs `predictions.mat` containing variable `predicted_dg`.
4. Download `predictions.mat` from the Files panel.

**Runtime:** ~10–15 minutes on CPU, ~3–5 minutes with GPU.  
**No training is performed.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'mne==1.7.0', 'scikit-learn', 'scipy', 'torch', 'numpy'])
print('Packages ready.')

In [ ]:
import zipfile, pathlib

zip_path = pathlib.Path('algorithm.zip')
assert zip_path.exists(), 'Please upload algorithm.zip first.'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('.')

print('Extracted:', sorted(str(p) for p in pathlib.Path('.').glob('**/*.pt')))

In [ ]:
import pickle, sys
import numpy as np
import scipy.io
import scipy.ndimage
import torch

sys.path.insert(0, '.')
from models import AutoEncoder1D
from predict_utils import preprocess_ecog, correct_time_delay, upsample_to_1000hz
from train_utils import predict_full, SMOOTH_SIGMA

DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
N_SUBJECTS  = 3
N_FREQS     = 40
WINDOW      = 256

CKPT_PATTERN   = 'checkpoints/subj{}_cnn_best_v4.pt'
SCALER_PATTERN = 'scalers/subj{}_ecog_scaler.pkl'

In [ ]:
assert pathlib.Path('truetest_data.mat').exists(), 'Please upload truetest_data.mat first.'

ecog_all = scipy.io.loadmat('truetest_data.mat')['truetest_data']

for subj in range(N_SUBJECTS):
    print(f'  Subject {subj+1}: ECoG shape = {ecog_all[subj, 0].shape}  (N×C)')

In [ ]:
predicted_dg = np.empty((N_SUBJECTS, 1), dtype=object)

for subj in range(N_SUBJECTS):
    print(f'\nSubject {subj + 1}')
    ecog_raw = ecog_all[subj, 0]   # (N, C)
    C = ecog_raw.shape[1]

    with open(SCALER_PATTERN.format(subj + 1), 'rb') as f:
        ecog_scaler = pickle.load(f)

    print(f'  Preprocessing ECoG {ecog_raw.shape} ...', flush=True)
    specs = preprocess_ecog(ecog_raw, ecog_scaler, C, N_FREQS)   # (C, F, T) @ 100 Hz
    print(f'  Spectrogram shape: {specs.shape}')

    model = AutoEncoder1D(n_electrodes=C, n_freqs=N_FREQS, n_out=5)
    model.load_state_dict(torch.load(CKPT_PATTERN.format(subj + 1), map_location=DEVICE))
    model.to(DEVICE).eval()
    print(f'  Loaded {CKPT_PATTERN.format(subj + 1)}')

    print('  Running inference ...', flush=True)
    raw_pred = predict_full(model, specs, DEVICE, WINDOW)                         # (T, 5) @ 100 Hz
    smooth   = scipy.ndimage.gaussian_filter1d(raw_pred.T, sigma=SMOOTH_SIGMA).T  # smooth
    aligned  = correct_time_delay(smooth)                                          # undo 200 ms shift
    pred_1k  = upsample_to_1000hz(aligned)                                         # (T*10, 5) @ 1000 Hz
    print(f'  Prediction shape: {pred_1k.shape}')

    predicted_dg[subj, 0] = pred_1k

scipy.io.savemat('predictions.mat', {'predicted_dg': predicted_dg})
print('\nSaved → predictions.mat')